# 📜 Smart Contract Assistant
**Groq `openai/gpt-oss-120b` · FAISS · LangChain · Gradio**

Run each cell in order. The final cell launches the Gradio UI with a public share link.

---
**Before starting:** Make sure you have your Groq API key from https://console.groq.com

## Step 1: Clone Repo & Install Dependencies

In [ ]:
# Clone the project (or upload folder via Files panel)
# If running from a zip upload, skip this cell
# !git clone https://github.com/YOUR_USERNAME/smart_contract_assistant.git
# %cd smart_contract_assistant

# Install all dependencies
!pip install -q -r requirements.txt

## Step 2: Set Your Groq API Key

In [ ]:
import os
from google.colab import userdata

# Option A: Use Colab Secrets (recommended — set via the 🔑 key icon in left sidebar)
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ GROQ_API_KEY loaded from Colab Secrets')
except Exception:
    # Option B: Paste directly (less secure)
    os.environ['GROQ_API_KEY'] = 'gsk_YOUR_KEY_HERE'
    print('⚠️  API key set manually — consider using Colab Secrets instead')

# Verify
assert os.environ.get('GROQ_API_KEY', '').startswith('gsk_'), \
    'Invalid Groq API key format. Keys start with gsk_'

## Step 3: Start the FastAPI Backend (in background)

In [ ]:
import subprocess
import sys
import time

# Start FastAPI backend in background
backend_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.api.server:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for startup
time.sleep(4)

# Verify it's running
import requests
try:
    resp = requests.get('http://localhost:8000/health', timeout=5)
    print('✅ Backend running:', resp.json())
except Exception as e:
    print('❌ Backend failed to start:', e)
    print(backend_process.stderr.read().decode())

## Step 4: Launch the Gradio UI
A **public share link** will be generated — click it to open the app.

In [ ]:
import sys
sys.path.insert(0, '.')

from frontend.app import build_ui

demo = build_ui()
demo.launch(
    share=True,         # Public URL for Colab
    server_name='0.0.0.0',
    server_port=7860,
    show_error=True,
    quiet=False,
)

---
## 🧪 Optional: Run Evaluation via Code

In [ ]:
import requests

# Make sure you've uploaded a document first via the UI!
test_questions = [
    'What are the parties involved in this contract?',
    'What are the payment terms?',
    'What is the contract duration?',
    'What are the termination conditions?',
]

resp = requests.post(
    'http://localhost:8000/evaluate',
    json={'questions': test_questions}
)
results = resp.json()

print(f"\n📊 Evaluation Summary ({results['num_questions']} questions)")
print(f"Faithfulness:       {results['avg_faithfulness']:.2%}")
print(f"Answer Relevance:   {results['avg_answer_relevance']:.2%}")
print(f"Citation Coverage:  {results['avg_citation_coverage']:.2%}")
print(f"Avg Latency:        {results['avg_latency_seconds']}s")

---
## 🛑 Stop Backend

In [ ]:
backend_process.terminate()
print('Backend stopped.')